# Análisis interno de ESM-2

## Actividades 4, 5, 6 y 7

Trabajo de Segundo Corte - Ciencia de Datos  
Universidad de Pamplona - Ingeniería de Sistemas

Este notebook desarrolla:

4. **Visualización de embeddings y hidden states:** embedding por residuo, embedding global, similitud coseno y PCA.
5. **Matrices de atención:** por capa y por cabeza.
6. **Masked Language Modeling:** predicción top-k de un aminoácido enmascarado.
7. **Usos reales y limitaciones de ESM-2.**

> Este trabajo no entrena el modelo ni construye un chatbot. El objetivo es analizar internamente un transformer científico preentrenado.


# Preparación del entorno


In [ ]:
%pip install -q torch transformers matplotlib scikit-learn pandas


# Importación de librerías


In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA

from transformers import AutoTokenizer, EsmModel, EsmForMaskedLM

print("PyTorch:", torch.__version__)


# Configuración del modelo y secuencias


In [ ]:
MODEL_NAME = "facebook/esm2_t6_8M_UR50D"

SEQUENCES = {
    "Original": "MKTAYIAKQRQISFVKSHFSRQDILD",
    "Mutada_Y_a_F": "MKTAFIAKQRQISFVKSHFSRQDILD",
    "Alterada": "DLIDQRSFHSSKVFSIQRQKAIYATKM",
}

SEQUENCES


# Cargar tokenizer y modelos

Se cargan dos versiones del modelo:

- EsmModel: para obtener hidden states y atenciones.
- EsmForMaskedLM: para masked language modeling.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

try:
    model = EsmModel.from_pretrained(MODEL_NAME, attn_implementation="eager")
    print("EsmModel cargado con attn_implementation='eager'.")
except TypeError:
    model = EsmModel.from_pretrained(MODEL_NAME)
    print("EsmModel cargado sin attn_implementation.")

model.eval()

mlm_model = EsmForMaskedLM.from_pretrained(MODEL_NAME)
mlm_model.eval()

print("Modelos cargados.")


# Función auxiliar: ejecutar el modelo sobre una secuencia


In [ ]:
def run_model(sequence, tokenizer, model):
    inputs = tokenizer(sequence, return_tensors="pt")
    with torch.no_grad():
        outputs = model(
            **inputs,
            output_hidden_states=True,
            output_attentions=True,
            return_dict=True,
        )
    return inputs, outputs


# Actividad 4: Embeddings y hidden states

Se extrae el `last_hidden_state` de cada secuencia. Cada fila corresponde al hidden state de un token.

El embedding global de la proteína se obtiene promediando los hidden states de los residuos, excluyendo tokens especiales.


In [ ]:
def get_global_embedding(outputs):
    last_hidden = outputs.last_hidden_state[0]   # (seq_len, hidden)
    residue_states = last_hidden[1:-1, :]        # excluir tokens especiales
    global_emb = residue_states.mean(dim=0).numpy()
    return global_emb, residue_states.numpy()


In [ ]:
global_embeddings = {}
residue_embeddings = {}

for name, seq in SEQUENCES.items():
    inputs, outputs = run_model(seq, tokenizer, model)
    g_emb, r_emb = get_global_embedding(outputs)
    global_embeddings[name] = g_emb
    residue_embeddings[name] = r_emb

    print(f"Secuencia: {name}")
    print(f"  last_hidden_state shape: {tuple(outputs.last_hidden_state.shape)}")
    print(f"  embedding por residuo shape: {r_emb.shape}")
    print(f"  embedding global shape: {g_emb.shape}")
    print()


## Similitud coseno entre embeddings globales

Se espera:

- Original vs Mutada: similitud alta (difieren en un aminoácido).
- Original vs Alterada: similitud menor (cambia el orden).


In [ ]:
names = list(global_embeddings.keys())
matrix = np.array([global_embeddings[n] for n in names])
sim = cosine_similarity(matrix)

df_sim = pd.DataFrame(sim, index=names, columns=names)
df_sim


## Proyección PCA de embeddings por residuo

Cada punto representa un aminoácido de alguna de las tres secuencias.
La visualización es solo una representación matemática reducida; no demuestra significado biológico por sí sola.


In [ ]:
all_vectors = []
all_labels = []

for name in names:
    vecs = residue_embeddings[name]
    all_vectors.append(vecs)
    all_labels.extend([name] * vecs.shape[0])

all_vectors = np.vstack(all_vectors)

pca = PCA(n_components=2)
coords = pca.fit_transform(all_vectors)

plt.figure(figsize=(8, 6))
for name in names:
    idx = [i for i, lab in enumerate(all_labels) if lab == name]
    plt.scatter(coords[idx, 0], coords[idx, 1], label=name, alpha=0.7)
plt.title("PCA de embeddings por residuo")
plt.xlabel("Componente 1")
plt.ylabel("Componente 2")
plt.legend()
plt.tight_layout()
plt.show()


# Actividad 5: Matrices de atención

La forma esperada de las atenciones es:

(batch, num_heads, sequence_length, sequence_length)

Cada matriz L x L indica cuánto atiende cada posición a las demás, para una capa y una cabeza específicas.

Limitación: la atención no prueba causalidad biológica.


In [ ]:
seq = SEQUENCES["Original"]
inputs, outputs = run_model(seq, tokenizer, model)

if outputs.attentions is None:
    print("Las atenciones no están disponibles en esta configuración.")
    print("Documenta el intento y centra el análisis en hidden states y logits.")
else:
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    num_layers = len(outputs.attentions)
    num_heads = outputs.attentions[0].shape[1]
    print("Número de capas con atención:", num_layers)
    print("Número de cabezas por capa:", num_heads)
    print("Forma atención capa 0:", tuple(outputs.attentions[0].shape))


## Función para graficar una matriz de atención


In [ ]:
def plot_attention(att_matrix, tokens, title):
    plt.figure(figsize=(8, 7))
    plt.imshow(att_matrix, aspect="auto")
    plt.colorbar(label="peso de atención")
    plt.xticks(range(len(tokens)), tokens, rotation=90)
    plt.yticks(range(len(tokens)), tokens)
    plt.title(title)
    plt.tight_layout()
    plt.show()


## Matriz de atención: capa temprana


In [ ]:
if outputs.attentions is not None:
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    att_early = outputs.attentions[0][0, 0].detach().cpu().numpy()
    plot_attention(att_early, tokens, "Atención - capa 0, cabeza 0")


## Matriz de atención: capa profunda


In [ ]:
if outputs.attentions is not None:
    deep_layer = len(outputs.attentions) - 1
    att_deep = outputs.attentions[deep_layer][0, 0].detach().cpu().numpy()
    plot_attention(att_deep, tokens, f"Atención - capa {deep_layer}, cabeza 0")


## Dos cabezas distintas de una misma capa


In [ ]:
if outputs.attentions is not None and outputs.attentions[0].shape[1] >= 2:
    att_h0 = outputs.attentions[0][0, 0].detach().cpu().numpy()
    att_h1 = outputs.attentions[0][0, 1].detach().cpu().numpy()
    plot_attention(att_h0, tokens, "Atención - capa 0, cabeza 0")
    plot_attention(att_h1, tokens, "Atención - capa 0, cabeza 1")


## Comparación de atención: original vs mutada


In [ ]:
seq_mut = SEQUENCES["Mutada_Y_a_F"]
inputs_mut, outputs_mut = run_model(seq_mut, tokenizer, model)

if outputs.attentions is not None and outputs_mut.attentions is not None:
    tokens_orig = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    tokens_mut = tokenizer.convert_ids_to_tokens(inputs_mut["input_ids"][0])

    att_orig = outputs.attentions[0][0, 0].detach().cpu().numpy()
    att_mut = outputs_mut.attentions[0][0, 0].detach().cpu().numpy()

    plot_attention(att_orig, tokens_orig, "Original - capa 0, cabeza 0")
    plot_attention(att_mut, tokens_mut, "Mutada - capa 0, cabeza 0")


# Actividad 6: Masked Language Modeling

Se oculta un aminoácido con el token de máscara y se obtienen los top-k aminoácidos más probables.

A diferencia de GPT, ESM-2 usa contexto bidireccional: observa ambos lados de la posición enmascarada.


In [ ]:
base_seq = "MKTAYIAKQRQISFVKSHFSRQDILD"
mask_token = tokenizer.mask_token

masked_sequence = base_seq.replace("Y", mask_token, 1)
print("Secuencia original:", base_seq)
print("Secuencia enmascarada:", masked_sequence)

masked_inputs = tokenizer(masked_sequence, return_tensors="pt")
mask_index = (masked_inputs["input_ids"] == tokenizer.mask_token_id).nonzero(as_tuple=True)[1]

with torch.no_grad():
    mlm_outputs = mlm_model(**masked_inputs)

logits = mlm_outputs.logits[0, mask_index, :]
topk = torch.topk(logits, k=5, dim=-1)

print("\nTop-5 tokens predichos para la posición enmascarada:")
for score, token_id in zip(topk.values[0], topk.indices[0]):
    token = tokenizer.convert_ids_to_tokens([token_id.item()])[0]
    print(f"  {token}  ->  score: {float(score):.4f}")

print("\nNota: el aminoácido original en esa posición era 'Y'.")


## Interpretación de la Actividad 6

- La posición enmascarada fue la primera 'Y' de la secuencia.
- El modelo devuelve aminoácidos candidatos con su puntaje.
- Esto no es generación tipo GPT: ESM-2 no genera token por token de izquierda a derecha.
- Al ser encoder-only, ESM-2 usa contexto bidireccional para predecir la posición oculta.


# Actividad 7: Usos reales y limitaciones

## Usos reales

1. Comparación de proteínas mediante embeddings.
2. Análisis de mutaciones.
3. Apoyo a predicción de estructura mediante modelos derivados como ESMFold.
4. Anotación funcional o clasificación posterior.
5. Búsqueda semántica de proteínas en bases vectoriales.
6. Priorización de variantes para ingeniería de proteínas.

## Limitaciones

- ESM-2 no descubre medicamentos automáticamente.
- No reemplaza experimentos de laboratorio.
- La atención no prueba causalidad biológica.
- Refleja sesgos de los datos de entrenamiento.
- Requiere recursos de cómputo según el tamaño del modelo.


# Conclusiones de las Actividades 4, 5, 6 y 7

1. Los embeddings de ESM-2 permiten comparar proteínas mediante similitud coseno.
2. La secuencia mutada se mantiene cercana a la original; la alterada se aleja más.
3. Las matrices de atención muestran patrones de relación, pero no causalidad biológica.
4. El masked language modeling demuestra el carácter bidireccional de ESM-2.
5. ESM-2 tiene usos reales en ciencia de datos aplicada a proteínas, con limitaciones claras.


# Referencias

- Vaswani, A. et al. (2017). *Attention Is All You Need*. NeurIPS.
- Devlin, J. et al. (2018). *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*.
- Radford, A. et al. (2018). *Improving Language Understanding by Generative Pre-Training*.
- Lin, Z. et al. (2023). *Evolutionary-scale prediction of atomic-level protein structure with a language model*. Science.
- Hugging Face Transformers Documentation: ESM models and model outputs.
